<a href="https://colab.research.google.com/github/Luidy006/fundos-imobiliarios/blob/main/Calculadora_foda_de_FIIs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Deepseek - Funcional: 17/08/2026

In [1]:
# @title Bloco 1: Importações e Configurações
import os, re, io, json, time, logging, warnings
import datetime as dt
from typing import Dict, List, Optional, Tuple
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    brier_score_loss, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
import matplotlib.pyplot as plt
import streamlit as st
from concurrent.futures import ThreadPoolExecutor, as_completed

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("FII")
logger.setLevel(logging.INFO)

# Suprime logs de bibliotecas externas
for lib in ["yfinance", "urllib3", "requests", "pandas_datareader", "investpy"]:
    logging.getLogger(lib).setLevel(logging.CRITICAL)
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0 Safari/537.36"
}

st.set_page_config(page_title="Recomendador de FIIs com Rede Bayesiana", layout="wide")
st.title("🏢 Recomendador Inteligente de Fundos Imobiliários")
st.markdown("---")

ModuleNotFoundError: No module named 'streamlit'

In [ ]:
# @title Bloco 2: Lista de FIIs (dinâmica via Fundamentus) e dados estáticos

@st.cache_data(ttl=3600, show_spinner=False)
def get_fii_tickers_from_fundamentus():
    """Obtém lista de FIIs da B3 via Fundamentus."""
    url = "https://www.fundamentus.com.br/fii_resultado.php"
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15, verify=False)
        if resp.status_code == 200:
            tables = pd.read_html(io.StringIO(resp.text))
            if tables:
                df = tables[0]
                if "Papel" in df.columns:
                    tickers = df["Papel"].astype(str).str.strip().str.upper()
                    tickers = tickers[tickers.str.endswith("11")].tolist()
                    return sorted(set(tickers))
    except Exception as e:
        logger.warning(f"Falha ao obter lista do Fundamentus: {e}")
    return None

# Lista estática de fallback (95 tickers conhecidos)
STATIC_TICKERS = [
    "HGLG11","KNRI11","XPML11","VISC11","BTLG11","MXRF11","CPTS11","RBRR11",
    "RBRF11","HFOF11","KNCR11","IRDM11","BRCO11","RECR11","SNCI11","XPSF11",
    "VRTA11","TGAR11","VSLH11","BCRI11","RZAK11","VINO11","RZTR11","SARE11",
    "ALZR11","BARI11","BCFF11","BMLC11","BRCR11","BTAL11","CARE11","CEOC11",
    "CVBI11","DEVA11","FIIB11","FINF11","GARE11","HABT11","HCTR11","HSLG11",
    "HGRU11","HGRE11","HGBS11","HGPO11","HGRS11","IDGR11","IFIX11","IRIM11",
    "JSRE11","KISU11","KNIP11","KNRE11","MALL11","MANA11","MFII11","MGFF11",
    "MORC11","MVFI11","NSLU11","PATL11","PORD11","PRSV11","RBED11","RBFF11",
    "RBBV11","RBRP11","RBRX11","RECT11","RFOF11","RMAI11","SADI11","SCPF11",
    "SDIL11","SEQR11","SHPH11","SPTW11","TBOF11","TEPP11","TFOF11","URPR11",
    "VCJR11","VGHF11","VGIP11","VGIR11","VILG11","VOTS11","VTRT11","WTSP11",
    "XPCI11","XPCM11","XPLG11","XPIN11","XPPR11","YCHY11","ZAVI11",
]

# Base estática realista para fallback fundamentalista
STATIC_FUNDAMENTALS = {
    "HGLG11": {"Preco": 145.11, "P/VP": 1.02, "DY": 7.8, "Vacancia": 5.2, "Liquidez": 8.5, "Segmento": "Logístico"},
    "KNRI11": {"Preco": 149.12, "P/VP": 1.10, "DY": 7.1, "Vacancia": 3.5, "Liquidez": 6.0, "Segmento": "Híbrido"},
    "XPML11": {"Preco": 100.84, "P/VP": 1.00, "DY": 8.5, "Vacancia": 6.0, "Liquidez": 12.0, "Segmento": "Shopping"},
    "VISC11": {"Preco": 101.07, "P/VP": 1.05, "DY": 7.5, "Vacancia": 5.0, "Liquidez": 5.2, "Segmento": "Shopping"},
    "BTLG11": {"Preco": 98.31, "P/VP": 0.98, "DY": 8.0, "Vacancia": 4.8, "Liquidez": 4.5, "Segmento": "Logístico"},
    "MXRF11": {"Preco": 9.24, "P/VP": 0.90, "DY": 11.2, "Vacancia": 7.5, "Liquidez": 9.8, "Segmento": "Recebíveis"},
    "CPTS11": {"Preco": 7.30, "P/VP": 0.85, "DY": 10.5, "Vacancia": 0.0, "Liquidez": 3.2, "Segmento": "Recebíveis"},
    "RBRR11": {"Preco": 73.00, "P/VP": 1.10, "DY": 7.0, "Vacancia": 6.5, "Liquidez": 3.8, "Segmento": "Híbrido"},
    "RBRF11": {"Preco": 6.65, "P/VP": 1.00, "DY": 7.8, "Vacancia": 5.5, "Liquidez": 2.9, "Segmento": "Fundo de Fundos"},
    "HFOF11": {"Preco": 6.12, "P/VP": 0.95, "DY": 8.7, "Vacancia": 6.2, "Liquidez": 2.0, "Segmento": "Fundo de Fundos"},
    "KNCR11": {"Preco": 105.24, "P/VP": 0.98, "DY": 8.2, "Vacancia": 0.0, "Liquidez": 7.0, "Segmento": "Recebíveis"},
    "IRDM11": {"Preco": 62.40, "P/VP": 0.92, "DY": 10.0, "Vacancia": 4.5, "Liquidez": 4.0, "Segmento": "Recebíveis"},
    "BRCO11": {"Preco": 110.00, "P/VP": 1.02, "DY": 7.6, "Vacancia": 7.0, "Liquidez": 5.5, "Segmento": "Logístico"},
    "RECR11": {"Preco": 82.00, "P/VP": 0.95, "DY": 8.0, "Vacancia": 5.8, "Liquidez": 2.5, "Segmento": "Híbrido"},
    "SNCI11": {"Preco": 95.00, "P/VP": 1.00, "DY": 7.4, "Vacancia": 6.0, "Liquidez": 1.8, "Segmento": "Híbrido"},
    "XPSF11": {"Preco": 90.00, "P/VP": 0.88, "DY": 9.8, "Vacancia": 2.5, "Liquidez": 1.2, "Segmento": "Recebíveis"},
    "VRTA11": {"Preco": 105.00, "P/VP": 1.05, "DY": 7.3, "Vacancia": 4.0, "Liquidez": 6.8, "Segmento": "Logístico"},
    "TGAR11": {"Preco": 120.00, "P/VP": 1.15, "DY": 6.8, "Vacancia": 3.8, "Liquidez": 5.0, "Segmento": "Logístico"},
    "VSLH11": {"Preco": 99.00, "P/VP": 0.97, "DY": 8.8, "Vacancia": 5.5, "Liquidez": 3.0, "Segmento": "Híbrido"},
    "BCRI11": {"Preco": 57.29, "P/VP": 0.90, "DY": 10.2, "Vacancia": 0.0, "Liquidez": 2.2, "Segmento": "Recebíveis"},
    "RZAK11": {"Preco": 92.50, "P/VP": 0.93, "DY": 9.5, "Vacancia": 3.0, "Liquidez": 1.5, "Segmento": "Recebíveis"},
    "VINO11": {"Preco": 85.00, "P/VP": 1.03, "DY": 8.1, "Vacancia": 4.5, "Liquidez": 1.0, "Segmento": "Híbrido"},
    "RZTR11": {"Preco": 98.00, "P/VP": 0.99, "DY": 8.3, "Vacancia": 6.8, "Liquidez": 2.0, "Segmento": "Logístico"},
    # ... (outros tickers podem ser adicionados, mas o fallback cobre apenas os conhecidos)
}
STATIC_FUNDAMENTALS = pd.DataFrame(STATIC_FUNDAMENTALS).T

INFO:FII:Lista dinâmica obtida: 553 FIIs
INFO:FII:Total de FIIs carregados: 553


In [ ]:
# @title Bloco 3: Coleta de dados macro e preços (com cache)

def try_get(url, timeout=15, retries=2, headers=None):
    """GET com retries."""
    headers = headers or HEADERS
    for _ in range(retries + 1):
        try:
            resp = requests.get(url, timeout=timeout, headers=headers, verify=False)
            if resp.status_code == 200 and resp.text.strip():
                return resp
        except Exception:
            time.sleep(1)
    return None

@st.cache_data(ttl=86400, show_spinner=False)
def fetch_bcb_series(codes: List[int], name: str) -> pd.Series:
    """Busca série macroeconômica no BCB com cache."""
    for code in codes:
        for fmt in ["json", "csv"]:
            url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{code}/dados?formato={fmt}"
            resp = try_get(url)
            if resp:
                try:
                    if fmt == "json":
                        data = resp.json()
                        if isinstance(data, dict):
                            data = [data]
                        df = pd.json_normalize(data)
                    else:
                        df = pd.read_csv(io.StringIO(resp.text), sep=';', engine='python')
                    if "data" in df.columns and "valor" in df.columns:
                        df["data"] = pd.to_datetime(df["data"], dayfirst=True, errors="coerce")
                        df["valor"] = pd.to_numeric(df["valor"].astype(str).str.replace(",", "."), errors="coerce")
                        s = df.set_index("data")["valor"].dropna().sort_index()
                        if len(s) > 1:
                            s.name = name
                            return s
                except Exception:
                    continue
    logger.warning(f"Não foi possível obter {name} do BCB. Usando série vazia.")
    return pd.Series(dtype=float, name=name)

def fetch_yahoo_chart_api(ticker: str, years: int = 12) -> pd.DataFrame:
    """Yahoo Finance Chart API v8."""
    end = pd.Timestamp.today().normalize()
    start = end - pd.DateOffset(years=years)
    p1, p2 = int(start.timestamp()), int(end.timestamp())
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}.SA?period1={p1}&period2={p2}&interval=1d"
    resp = try_get(url, headers={"User-Agent": "Mozilla/5.0"})
    if resp:
        data = resp.json()
        if "chart" in data and "result" in data["chart"]:
            res = data["chart"]["result"][0]
            timestamps = res.get("timestamp", [])
            closes = res["indicators"]["quote"][0].get("close", [])
            volumes = res["indicators"]["quote"][0].get("volume", [])
            df = pd.DataFrame({"Date": pd.to_datetime(timestamps, unit="s"), "Close": closes, "Volume": volumes})
            df = df.dropna(subset=["Close"])
            if not df.empty:
                return df
    raise ValueError("Yahoo Chart API falhou")

def fetch_stooq_csv(ticker: str, years: int = 12) -> pd.DataFrame:
    """Stooq CSV."""
    url = f"https://stooq.com/q/d/l/?s={ticker.lower()}.sa&i=d"
    resp = try_get(url)
    if resp and not resp.text.startswith("<"):
        df = pd.read_csv(io.StringIO(resp.text))
        df.columns = [c.strip() for c in df.columns]
        df["Date"] = pd.to_datetime(df["Date"])
        start = pd.Timestamp.today().normalize() - pd.DateOffset(years=years)
        df = df[df["Date"] >= start].reset_index(drop=True)
        if not df.empty:
            return df[["Date", "Close", "Volume"]]
    raise ValueError("Stooq CSV falhou")

def fetch_yfinance_lib(ticker: str, years: int = 12) -> pd.DataFrame:
    """yfinance."""
    import yfinance as yf
    data = yf.download(ticker + ".SA", period=f"{years}y", auto_adjust=False, progress=False)
    if data.empty:
        raise ValueError("yfinance vazio")
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.droplevel(1)
    data = data.reset_index()
    if "Date" not in data.columns and "Datetime" in data.columns:
        data.rename(columns={"Datetime": "Date"}, inplace=True)
    if "Adj Close" in data.columns and "Close" not in data.columns:
        data.rename(columns={"Adj Close": "Close"}, inplace=True)
    return data[["Date", "Close", "Volume"]]

def fetch_price_history(ticker: str) -> pd.DataFrame:
    """Tenta Yahoo Chart, Stooq, yfinance (retorna DataFrame ou None)."""
    methods = [
        ("Yahoo Chart", fetch_yahoo_chart_api),
        ("Stooq", fetch_stooq_csv),
        ("yfinance", fetch_yfinance_lib),
    ]
    for name, method in methods:
        try:
            df = method(ticker)
            if df is not None and len(df) > 10:
                return df
        except Exception:
            continue
    return None

@st.cache_data(ttl=86400, show_spinner=False)
def fetch_fundamental_data(tickers: List[str]) -> pd.DataFrame:
    """Tenta obter dados fundamentalistas via StatusInvest (simplificado)."""
    dados = {}
    for t in tickers:
        try:
            url = f"https://statusinvest.com.br/fundos-imobiliarios/{t.lower()}"
            resp = requests.get(url, headers=HEADERS, timeout=10, verify=False)
            if resp.status_code == 200:
                m_price = re.search(r'"price":\s*([0-9.]+)', resp.text)
                m_dy = re.search(r'"dividendYield":\s*([0-9.]+)', resp.text)
                if m_price and m_dy:
                    dados[t] = {
                        "Preco": float(m_price.group(1)),
                        "DY": float(m_dy.group(1)) * 100,
                        "P/VP": np.nan,
                        "Vacancia": np.nan,
                        "Liquidez": np.nan,
                        "Segmento": "Coletado",
                    }
                    continue
        except Exception:
            pass
        if t in STATIC_FUNDAMENTALS.index:
            dados[t] = STATIC_FUNDAMENTALS.loc[t].to_dict()
        else:
            dados[t] = {"Preco": 100.0, "P/VP": 1.0, "DY": 8.0, "Vacancia": 5.0, "Liquidez": 1.0, "Segmento": "Desconhecido"}
    return pd.DataFrame(dados).T

@st.cache_data(ttl=86400, show_spinner=False)
def load_market_data():
    """Coleta série macro e preços de todos os FIIs."""
    logger.info("Coletando séries macroeconômicas...")
    selic = fetch_bcb_series([432, 4189, 1178], "selic")
    ipca12 = fetch_bcb_series([13522], "ipca12")
    ipca_mensal = fetch_bcb_series([433], "ipca")

    tickers_dinamicos = get_fii_tickers_from_fundamentus()
    if tickers_dinamicos and len(tickers_dinamicos) > 100:
        TICKERS = tickers_dinamicos
    else:
        TICKERS = sorted(set([t.upper() for t in STATIC_TICKERS if isinstance(t, str) and t.endswith("11")]))

    logger.info("Coletando preços dos FIIs em paralelo...")
    price_hist = {}
    erros = []
    with ThreadPoolExecutor(max_workers=10) as executor:
        future_to_ticker = {executor.submit(fetch_price_history, t): t for t in TICKERS}
        for future in as_completed(future_to_ticker):
            ticker = future_to_ticker[future]
            try:
                df = future.result()
                if df is not None and len(df) > 10:
                    price_hist[ticker] = df
                else:
                    erros.append(ticker)
            except Exception:
                erros.append(ticker)

    if erros:
        logger.warning(f"Falha na coleta de preços para {len(erros)} FIIs: {erros[:10]}...")
    TICKERS_VALIDOS = list(price_hist.keys())
    logger.info(f"FIIs com dados reais: {len(TICKERS_VALIDOS)}")
    return selic, ipca12, ipca_mensal, price_hist, TICKERS_VALIDOS

INFO:FII:Coletando séries macroeconômicas...
INFO:FII:Série selic (código 4189) coletada: 481 obs
INFO:FII:Série ipca12 (código 13522) coletada: 548 obs
INFO:FII:Série ipca (código 433) coletada: 559 obs
INFO:FII:Coletando preços dos FIIs em paralelo...
INFO:FII:FIIs com dados reais: 436


In [ ]:
# @title Bloco 4: Features e Dataset Histórico (multi-horizonte)

FEATURES_CONT = ["Momentum", "Volatilidade", "Liquidez"]
FEATURES_CAT = ["Regime", "Mercado"]
HORIZONS_MONTHS = [3, 12, 36, 120]  # 3 meses, 1 ano, 3 anos, 10 anos
TARGET_COL = "Recomendacao"

def get_quantile_edges(series, q=4):
    try:
        jitter = series + np.random.normal(0, 1e-8, size=len(series))
        edges = pd.qcut(jitter, q=q, duplicates="drop", retbins=True)[1]
        if len(edges) < q + 1:
            edges = np.histogram_bin_edges(series, bins=q)
        return edges
    except:
        return np.histogram_bin_edges(series, bins=q)

def classify_regime(selic_val, selic_change_6m, ipca12_val):
    ipca_high = ipca12_val > 6.0
    ipca_low = ipca12_val < 4.5
    selic_rising = selic_change_6m > 0.5
    if ipca_high and selic_rising:
        return "estagflacao"
    elif (ipca_low or ipca12_val < 6.0) and not selic_rising:
        return "crescimento"
    return "neutro"

def classify_market(ret_3m, vol, vol_median):
    if ret_3m > 0.03 and vol < vol_median:
        return "bull"
    elif ret_3m < -0.03 or vol > 1.25 * vol_median:
        return "bear"
    return "neutro"

def build_historical_dataset(price_hist, selic, ipca12, ipca_mensal):
    """
    Cria dataset com features e targets para vários horizontes.
    Para cada data/ticker, calcula retorno futuro de 3, 12, 36 e 120 meses.
    """
    close_pivot = pd.DataFrame({t: df.set_index("Date")["Close"] for t, df in price_hist.items()})
    volume_pivot = pd.DataFrame({t: df.set_index("Date")["Volume"] for t, df in price_hist.items()})
    close_pivot = close_pivot.sort_index().ffill()
    volume_pivot = volume_pivot.sort_index().ffill()
    close_pivot = close_pivot.dropna(thresh=max(10, int(len(price_hist)*0.5)))
    volume_pivot = volume_pivot.loc[close_pivot.index]

    daily_ret = close_pivot.pct_change(fill_method=None)
    count_valid = daily_ret.notna().sum(axis=1)
    market_daily_ret = daily_ret.mean(axis=1)
    market_daily_ret[count_valid < 5] = np.nan
    market_index = (1 + market_daily_ret.fillna(0)).cumprod()
    vol_daily = daily_ret.rolling(63).std() * np.sqrt(252)
    market_vol_daily = market_daily_ret.rolling(63).std() * np.sqrt(252)
    turnover_daily = close_pivot * volume_pivot
    liq_daily = turnover_daily.rolling(63).mean()

    try:
        monthly_close = close_pivot.resample("ME").last()
        monthly_market_index = market_index.resample("ME").last()
        monthly_vol = vol_daily.resample("ME").last()
        monthly_market_vol = market_vol_daily.resample("ME").last()
        monthly_liq = liq_daily.resample("ME").last()
    except:
        monthly_close = close_pivot.resample("M").last()
        monthly_market_index = market_index.resample("M").last()
        monthly_vol = vol_daily.resample("M").last()
        monthly_market_vol = market_vol_daily.resample("M").last()
        monthly_liq = liq_daily.resample("M").last()

    selic_monthly = selic.reindex(monthly_close.index, method="ffill")
    ipca12_monthly = ipca12.reindex(monthly_close.index, method="ffill")
    ipca_mensal_monthly = ipca_mensal.reindex(monthly_close.index, method="ffill")
    selic_change_6m = selic_monthly - selic_monthly.shift(6)

    rows = []
    for ticker in monthly_close.columns:
        for date in monthly_close.index:
            price = monthly_close.loc[date, ticker]
            if pd.isna(price):
                continue
            mom_3m = price / monthly_close[ticker].shift(3).loc[date] - 1
            vol = monthly_vol.loc[date, ticker]
            liq = monthly_liq.loc[date, ticker]
            mkt_ret_3m = monthly_market_index.loc[date] / monthly_market_index.shift(3).loc[date] - 1
            mkt_vol = monthly_market_vol.loc[date]
            selic_val = selic_monthly.loc[date]
            ipca12_val = ipca12_monthly.loc[date]
            ipca_m_val = ipca_mensal_monthly.loc[date]
            selic_chg = selic_change_6m.loc[date]
            if pd.isna(mom_3m) or pd.isna(vol) or pd.isna(liq) or pd.isna(mkt_ret_3m) or pd.isna(mkt_vol):
                continue
            if pd.isna(selic_val) or pd.isna(ipca12_val):
                continue
            regime = classify_regime(selic_val, selic_chg, ipca12_val)
            mkt_vol_median = market_vol_daily.rolling(252).median().resample("ME").last()
            mkt_cond = classify_market(mkt_ret_3m, mkt_vol, mkt_vol_median.loc[date] if date in mkt_vol_median.index else 0.20)

            row = {
                "Date": date, "Ticker": ticker, "Momentum": mom_3m,
                "Volatilidade": vol, "Liquidez": liq, "Retorno_Mercado": mkt_ret_3m,
                "Volatilidade_Mercado": mkt_vol, "Selic": selic_val, "IPCA_12M": ipca12_val,
                "IPCA_Mensal": ipca_m_val, "Regime": regime, "Mercado": mkt_cond,
            }
            # Targets para cada horizonte
            for h in HORIZONS_MONTHS:
                future_price = monthly_close[ticker].shift(-h).loc[date]
                if pd.isna(future_price):
                    row[f"Future_Return_{h}M"] = np.nan
                    row[f"Risk_Free_{h}M"] = np.nan
                    row[f"Target_{h}M"] = np.nan
                else:
                    future_ret = future_price / price - 1
                    risk_free = (1 + selic_val / 100) ** (h / 12) - 1
                    row[f"Future_Return_{h}M"] = future_ret
                    row[f"Risk_Free_{h}M"] = risk_free
                    row[f"Target_{h}M"] = 1 if future_ret > risk_free else 0
            rows.append(row)

    df = pd.DataFrame(rows)
    logger.info(f"Dataset histórico gerado: {len(df)} linhas")
    return df

INFO:FII:Dataset histórico gerado: 11336 linhas
INFO:FII:Dataset acumulado salvo: 11336 linhas
INFO:FII:Dataset discreto: 11327 linhas


In [ ]:
# @title Bloco 5: Armazenamento de interações (feedbacks) e avaliação temporal

INTERACTIONS_PATH = "user_interactions.csv"
RAW_DATASET_PATH = "fii_raw_dataset.csv"

def load_interactions():
    if os.path.exists(INTERACTIONS_PATH):
        try:
            return pd.read_csv(INTERACTIONS_PATH, parse_dates=["Date_At_Rec", "Target_Date"])
        except:
            return pd.DataFrame()
    return pd.DataFrame(columns=[
        "Timestamp", "Ticker", "Capital", "Profile", "Objective", "Horizon_Months",
        "Price_At_Rec", "Date_At_Rec", "Target_Date", "Predicted_Prob",
        "Recommended", "Actual_Return", "Risk_Free_Return", "Outcome", "Evaluated"
    ])

def save_interactions(df):
    df.to_csv(INTERACTIONS_PATH, index=False)
    logger.info(f"Interações salvas: {len(df)}")

def load_raw_dataset():
    if os.path.exists(RAW_DATASET_PATH):
        try:
            return pd.read_csv(RAW_DATASET_PATH, parse_dates=["Date"])
        except:
            return pd.DataFrame()
    return pd.DataFrame()

def save_raw_dataset(df):
    df.to_csv(RAW_DATASET_PATH, index=False)
    logger.info(f"Dataset bruto salvo: {len(df)} linhas")

def get_price_on_date(ticker, date, price_hist):
    """Retorna o preço de fechamento mais recente até a data especificada."""
    df = price_hist.get(ticker)
    if df is None:
        return None
    df = df[df["Date"] <= date]
    if df.empty:
        return None
    return df["Close"].iloc[-1]

def evaluate_matured_interactions(interactions, price_hist, selic):
    """Avalia interações cuja data alvo já passou. Retorna DataFrame atualizado."""
    today = pd.Timestamp.today().normalize()
    if interactions.empty:
        return interactions

    for idx, row in interactions.iterrows():
        if row["Evaluated"]:
            continue
        target_date = row["Target_Date"]
        if target_date <= today:
            ticker = row["Ticker"]
            rec_date = row["Date_At_Rec"]
            price_rec = row["Price_At_Rec"]
            price_target = get_price_on_date(ticker, target_date, price_hist)
            if price_target is None:
                # tenta usar o último preço disponível se target_date >= última data
                df = price_hist.get(ticker)
                if df is not None and not df.empty:
                    if df["Date"].max() >= target_date:
                        price_target = df[df["Date"] >= target_date]["Close"].iloc[0]
                    else:
                        continue  # ainda não há dados suficientes
            if price_target is None:
                continue
            actual_ret = price_target / price_rec - 1
            # taxa livre de risco no período
            selic_at_rec = selic.reindex(pd.Index([rec_date]), method="ffill")
            if len(selic_at_rec) > 0 and not pd.isna(selic_at_rec.iloc[0]):
                selic_val = selic_at_rec.iloc[0]
            else:
                selic_val = 0.0
            h = row["Horizon_Months"]
            risk_free = (1 + selic_val / 100) ** (h / 12) - 1
            outcome = 1 if actual_ret > risk_free else 0
            interactions.at[idx, "Actual_Return"] = actual_ret
            interactions.at[idx, "Risk_Free_Return"] = risk_free
            interactions.at[idx, "Outcome"] = outcome
            interactions.at[idx, "Evaluated"] = True
            logger.info(f"Avaliada interação {ticker} de {rec_date.date()} -> {target_date.date()} | Retorno {actual_ret:.2%} | Outcome {outcome}")
    return interactions

def merge_feedback_into_raw(raw, interactions, price_hist):
    """Adiciona/atualiza os targets no dataset bruto com base nos feedbacks avaliados."""
    if interactions.empty:
        return raw
    if raw.empty:
        return raw
    feedback = interactions[interactions["Evaluated"]].copy()
    if feedback.empty:
        return raw

    # Converte colunas de target para numéricas se ainda não forem
    for h in HORIZONS_MONTHS:
        col = f"Target_{h}M"
        if col not in raw.columns:
            raw[col] = np.nan
        raw[col] = pd.to_numeric(raw[col], errors="coerce")

    for _, row in feedback.iterrows():
        ticker = row["Ticker"]
        rec_date = row["Date_At_Rec"]
        h = row["Horizon_Months"]
        outcome = row["Outcome"]
        mask = (raw["Ticker"] == ticker) & (raw["Date"] == rec_date)
        if mask.any():
            raw.loc[mask, f"Target_{h}M"] = outcome
        else:
            # Se não existe a linha, cria uma a partir das features atuais?
            # Neste app, as recomendações são geradas a partir de dados atuais já presentes no raw.
            pass
    return raw

INFO:FII:Modelos treinados com sucesso.
INFO:FII:Calibração isotônica ajustada.


In [ ]:
# @title Bloco 6: Discretização e Treinamento da Rede Bayesiana

STATE_LABELS = {
    "Momentum": ["muito_baixo", "baixo", "alto", "muito_alto"],
    "Volatilidade": ["muito_baixa", "baixa", "alta", "muito_alta"],
    "Liquidez": ["muito_baixa", "baixa", "alta", "muito_alta"],
    "Regime": ["crescimento", "estagflacao", "neutro"],
    "Mercado": ["bull", "bear", "neutro"],
    "Recomendacao": ["nao", "sim"],
}

def discretize_dataframe_for_horizon(raw_df, edges, horizon_months):
    """Discretiza o dataset para um horizonte específico."""
    target_col = f"Target_{horizon_months}M"
    if target_col not in raw_df.columns:
        return pd.DataFrame()
    df = raw_df.copy()
    # Remove linhas com target NaN
    df = df.dropna(subset=[target_col])
    for col in FEATURES_CONT:
        labels = STATE_LABELS[col]
        df[col] = pd.cut(df[col], bins=edges[col], labels=labels, include_lowest=True)
    for col in FEATURES_CAT:
        df[col] = df[col].astype(str)
    df[TARGET_COL] = df[target_col].map({1: "sim", 0: "nao"})
    cols = ["Date", "Ticker"] + FEATURES_CONT + FEATURES_CAT + [TARGET_COL]
    return df[cols].dropna(subset=FEATURES_CONT + FEATURES_CAT + [TARGET_COL])

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

def build_bn_model(data):
    model = DiscreteBayesianNetwork([
        ("Regime", "Mercado"),
        ("Regime", "Recomendacao"),
        ("Mercado", "Recomendacao"),
        ("Momentum", "Recomendacao"),
        ("Volatilidade", "Recomendacao"),
        ("Liquidez", "Recomendacao"),
    ])
    nodes = FEATURES_CONT + FEATURES_CAT + [TARGET_COL]
    model.add_nodes_from(nodes)
    training_data = data[nodes].copy()
    for col in nodes:
        training_data[col] = training_data[col].astype("category")
    estimator = BayesianEstimator(model, training_data)
    for node in model.nodes():
        parents = list(model.get_parents(node))
        cpd = estimator.estimate_cpd(node, prior_type="BDeu", equivalent_sample_size=10)
        model.add_cpds(cpd)
    if not model.check_model():
        raise ValueError("Modelo inconsistente após estimação das CPDs")
    return model

def get_bn_probs(model, df):
    infer = VariableElimination(model)
    probs = []
    for _, row in df.iterrows():
        evidence = {col: row[col] for col in FEATURES_CONT + FEATURES_CAT}
        try:
            q = infer.query(variables=[TARGET_COL], evidence=evidence, show_progress=False)
            state_names = list(q.state_names[TARGET_COL])
            idx = state_names.index("sim")
            probs.append(q.values[idx])
        except Exception:
            probs.append(0.5)
    return np.array(probs)

def train_models_for_all_horizons(raw_all, edges):
    """Treina um modelo BN e calibração isotônica para cada horizonte."""
    models = {}
    calibrations = {}
    dates_sorted = sorted(raw_all["Date"].unique())
    n = len(dates_sorted)
    if n < 10:
        st.error("Dados históricos insuficientes para treinar modelos.")
        return {}, {}
    train_dates = dates_sorted[:int(0.6*n)]
    val_dates = dates_sorted[int(0.6*n):int(0.8*n)]

    for h in HORIZONS_MONTHS:
        discrete_h = discretize_dataframe_for_horizon(raw_all, edges, h)
        if discrete_h.empty:
            continue
        train_df = discrete_h[discrete_h["Date"].isin(train_dates)]
        val_df = discrete_h[discrete_h["Date"].isin(val_dates)]
        if train_df.empty or val_df.empty:
            continue
        try:
            model_eval = build_bn_model(train_df)
            val_probs = get_bn_probs(model_eval, val_df)
            calib_model = IsotonicRegression(out_of_bounds="clip")
            y_val = val_df[TARGET_COL].map({"sim": 1, "nao": 0}).values
            calib_model.fit(val_probs, y_val)
            # Modelo final com todos os dados
            final_model = build_bn_model(discrete_h)
            models[h] = final_model
            calibrations[h] = calib_model
        except Exception as e:
            logger.warning(f"Falha ao treinar modelo para horizonte {h}M: {e}")
    return models, calibrations

def get_current_predictions(models, calibrations, current_discrete, horizon_months):
    """Retorna probabilidade calibrada para o horizonte especificado."""
    if horizon_months not in models:
        return 0.5
    probs = get_bn_probs(models[horizon_months], current_discrete)
    if horizon_months in calibrations:
        probs = calibrations[horizon_months].predict(probs)
    return probs

Probabilidades Preditas pelo Modelo Final (Top 20):


,Ticker,Probabilidade_BN
365,RELG11,54.53%
277,PQAG11,54.53%
125,TRXF11,50.00%
199,KNCR11,50.00%
236,KCRE11,30.03%
140,CJCT11,30.03%
425,ITIT11,30.03%
381,IAGR11,28.61%
116,TSER11,28.61%
186,FISC11,28.61%


In [ ]:
# @title Bloco 7: Simulação de Monte Carlo e Score de Adequação

def project_price_monte_carlo(current_price, base_prob, vol_annual, months=12, n_sim=2000, seed=42):
    rng = np.random.default_rng(seed)
    sigma_m = vol_annual / np.sqrt(12)
    annual_drift = (base_prob - 0.5) * 0.30
    drift_m = annual_drift / 12
    scenarios = {
        "bull": {"prob": 0.35, "drift": 0.010},
        "neutral": {"prob": 0.40, "drift": 0.000},
        "bear": {"prob": 0.25, "drift": -0.015},
    }
    expected_prices = []
    prob_neg_components = []
    for scen in scenarios.values():
        scen_drift = drift_m + scen["drift"]
        z = rng.normal(size=(n_sim, months))
        monthly_ret = scen_drift + sigma_m * z
        price_paths = current_price * np.exp(np.cumsum(monthly_ret, axis=1))
        final_prices = price_paths[:, -1]
        expected_prices.append(scen["prob"] * np.mean(final_prices))
        prob_neg_components.append(scen["prob"] * np.mean(final_prices < current_price))
    expected_price = np.sum(expected_prices)
    prob_neg_total = np.sum(prob_neg_components)
    ret_esperado = expected_price / current_price - 1
    return expected_price, ret_esperado, prob_neg_total

def adjust_score(row, current_df, profile, objective, horizon):
    base = row["Base_Prob"]
    pct_vol = current_df["Volatilidade"].rank(pct=True).loc[row.name]
    pct_mom = current_df["Momentum"].rank(pct=True).loc[row.name]
    pct_liq = current_df["Liquidez"].rank(pct=True).loc[row.name]
    pct_dy = current_df["DY"].rank(pct=True).loc[row.name]
    pct_pvp = current_df["P/VP"].rank(pct=True).loc[row.name]
    pct_vac = current_df["Vacancia"].rank(pct=True).loc[row.name]

    adj = 0.0
    if objective == "renda":
        adj += 0.15 * pct_dy + 0.05 * (1 - pct_vac)
    elif objective == "valorização":
        adj += 0.10 * pct_mom + 0.10 * (1 - pct_pvp)
    elif objective == "equilíbrio":
        adj += 0.05 * pct_dy + 0.05 * pct_mom

    if profile == "conservador":
        adj -= 0.15 * pct_vol
    elif profile == "moderado":
        adj -= 0.05 * pct_vol
    elif profile == "agressivo":
        adj += 0.10 * pct_vol

    if horizon == "curto":
        adj -= 0.10 * pct_vol + 0.10 * (1 - pct_liq)
    elif horizon == "longo":
        adj += 0.05 * pct_vol

    return float(np.clip(base + adj, 0, 1))

Análise para perfil conservador, objetivo renda, horizonte médio.



,Segmento,Preco,P/VP,DY,Vacancia,Liquidez,Div_Anual_Estimado,Adequação,Preco_Projetado,Retorno_Esperado,Prob_Queda
XPSF11,Recebíveis,5.750000,0.880000,9.8%,2.5%,R$ 1.20 mi,R$ 0.56,64.97%,R$ 6.04,+5.11%,47.06%
KNCR11,Recebíveis,105.239998,0.980000,8.2%,0.0%,R$ 7.00 mi,R$ 8.63,64.95%,R$ 110.51,+5.01%,46.91%
CPTS11,Recebíveis,7.300000,0.850000,10.5%,0.0%,R$ 3.20 mi,R$ 0.77,64.01%,R$ 7.74,+5.99%,47.61%
RZAK11,Recebíveis,74.510002,0.930000,9.5%,3.0%,R$ 1.50 mi,R$ 7.08,62.75%,R$ 79.09,+6.15%,48.35%
BCRI11,Recebíveis,57.290001,0.900000,10.2%,0.0%,R$ 2.20 mi,R$ 5.84,62.51%,R$ 61.12,+6.68%,48.48%


In [ ]:
# @title Bloco 8: Lógica principal de preparação e execução da análise

def main():
    # Sidebar com entradas do usuário
    st.sidebar.header("🔍 Parâmetros da Análise")
    capital = st.sidebar.number_input("Capital para investir (R$)", min_value=1000, value=10000, step=1000)
    profile = st.sidebar.selectbox("Perfil do investidor", ["conservador", "moderado", "agressivo"], index=1)
    objective = st.sidebar.selectbox("Objetivo", ["renda", "valorização", "equilíbrio"], index=0)
    horizon = st.sidebar.selectbox("Horizonte de investimento", ["curto", "médio", "longo"], index=1)
    n_fiis = st.sidebar.slider("Quantidade de FIIs no ranking", min_value=5, max_value=30, value=10, step=1)
    run_button = st.sidebar.button("🚀 Executar Análise", type="primary")

    # Carregar dados (com cache)
    with st.spinner("Carregando dados de mercado (pode demorar alguns minutos na primeira vez)..."):
        selic, ipca12, ipca_mensal, price_hist, tickers_validos = load_market_data()

    # Construir dataset bruto (com cache)
    if "raw_all" not in st.session_state:
        raw_file = load_raw_dataset()
        if raw_file.empty:
            raw_all = build_historical_dataset(price_hist, selic, ipca12, ipca_mensal)
            save_raw_dataset(raw_all)
        else:
            raw_all = raw_file
        st.session_state.raw_all = raw_all
    else:
        raw_all = st.session_state.raw_all

    # Carregar interações
    interactions = load_interactions()

    # Avaliar predições vencidas
    if not interactions.empty:
        interactions = evaluate_matured_interactions(interactions, price_hist, selic)
        save_interactions(interactions)
        raw_all = merge_feedback_into_raw(raw_all, interactions, price_hist)
        save_raw_dataset(raw_all)
        st.session_state.raw_all = raw_all

    # Preparar edges de discretização
    dates_sorted = sorted(raw_all["Date"].unique())
    cutoff_date = dates_sorted[int(0.8 * len(dates_sorted))]
    train_for_bins = raw_all[raw_all["Date"] <= cutoff_date]
    edges = {col: get_quantile_edges(train_for_bins[col], q=4) for col in FEATURES_CONT}

    # Treinar modelos (em cache na sessão)
    if "models" not in st.session_state or st.sidebar.button("🔄 Retreinar Modelos"):
        with st.spinner("Treinando redes bayesianas..."):
            models, calibrations = train_models_for_all_horizons(raw_all, edges)
            st.session_state.models = models
            st.session_state.calibrations = calibrations
    models = st.session_state.models
    calibrations = st.session_state.calibrations

    if not models:
        st.error("Não foi possível treinar nenhum modelo. Verifique os dados.")
        return

    # Preparar dados atuais
    fundamental_df = fetch_fundamental_data(tickers_validos)
    current_raw = raw_all[raw_all["Ticker"].isin(tickers_validos)].sort_values("Date").groupby("Ticker").tail(1).copy()
    current_raw = current_raw.set_index("Ticker")

    # Imputar features ausentes
    missing = set(tickers_validos) - set(current_raw.index)
    if missing:
        median_vals = current_raw[FEATURES_CONT].median()
        mode_vals = current_raw[FEATURES_CAT].mode().iloc[0]
        for t in missing:
            current_raw.loc[t] = {
                "Date": current_raw["Date"].max(),
                "Momentum": median_vals["Momentum"],
                "Volatilidade": median_vals["Volatilidade"],
                "Liquidez": median_vals["Liquidez"],
                "Retorno_Mercado": current_raw["Retorno_Mercado"].median(),
                "Volatilidade_Mercado": current_raw["Volatilidade_Mercado"].median(),
                "Selic": current_raw["Selic"].median(),
                "IPCA_12M": current_raw["IPCA_12M"].median(),
                "IPCA_Mensal": current_raw["IPCA_Mensal"].median(),
                "Regime": mode_vals["Regime"],
                "Mercado": mode_vals["Mercado"],
            }

    current_discrete = pd.DataFrame(index=current_raw.index)
    for col in FEATURES_CONT:
        current_discrete[col] = pd.cut(
            current_raw[col].clip(edges[col][0], edges[col][-1]),
            bins=edges[col], labels=STATE_LABELS[col], include_lowest=True
        )
    for col in FEATURES_CAT:
        current_discrete[col] = current_raw[col].apply(
            lambda x: x if x in STATE_LABELS[col] else STATE_LABELS[col][-1]
        )

    # Probabilidade base para o horizonte selecionado
    horizon_months = {"curto": 12, "médio": 36, "longo": 120}[horizon]
    bn_probs = get_current_predictions(models, calibrations, current_discrete, horizon_months)

    # Score fundamentalista
    fund_percentiles = pd.DataFrame({
        "pct_pvp": fundamental_df["P/VP"].rank(pct=True),
        "pct_dy": fundamental_df["DY"].rank(pct=True),
        "pct_vac": fundamental_df["Vacancia"].rank(pct=True),
        "pct_liq": fundamental_df["Liquidez"].rank(pct=True),
    })
    score_fund = (0.25*(1-fund_percentiles["pct_pvp"]) + 0.35*fund_percentiles["pct_dy"] +
                  0.20*(1-fund_percentiles["pct_vac"]) + 0.20*fund_percentiles["pct_liq"]).clip(0,1)

    # Probabilidade base final
    base_prob = (0.7*bn_probs + 0.3*score_fund).clip(0,1)

    # Preço real
    preco_mercado = {t: price_hist[t]["Close"].iloc[-1] for t in tickers_validos}

    # DataFrame consolidado
    current_df = pd.DataFrame(index=tickers_validos)
    current_df["Preco"] = [preco_mercado[t] for t in tickers_validos]
    current_df["P/VP"] = fundamental_df["P/VP"]
    current_df["DY"] = fundamental_df["DY"]
    current_df["Vacancia"] = fundamental_df["Vacancia"]
    current_df["Liquidez"] = fundamental_df["Liquidez"]
    current_df["Segmento"] = fundamental_df["Segmento"]
    current_df["Momentum"] = current_raw["Momentum"]
    current_df["Volatilidade"] = current_raw["Volatilidade"]
    current_df["BN_Prob"] = bn_probs
    current_df["Score_Fund"] = score_fund
    current_df["Base_Prob"] = base_prob

    # Ajuste pelo perfil/objetivo/horizonte
    scores = current_df.apply(lambda r: adjust_score(r, current_df, profile, objective, horizon), axis=1)
    current_df["Adequação"] = scores
    current_df = current_df.sort_values("Adequação", ascending=False)

    # Selecionar top N
    top_df = current_df.head(n_fiis).copy()

    # Projeções para cada FII selecionado
    projections = []
    for t in top_df.index:
        price = top_df.loc[t, "Preco"]
        vol = top_df.loc[t, "Volatilidade"]
        bp = top_df.loc[t, "Base_Prob"]
        exp_price, ret_esp, prob_neg = project_price_monte_carlo(price, bp, vol, months=horizon_months)
        projections.append({
            "Ticker": t,
            "Preco_Projetado": exp_price,
            "Retorno_Esperado": ret_esp,
            "Prob_Queda": prob_neg,
        })
    proj_df = pd.DataFrame(projections).set_index("Ticker")
    top_df = top_df.join(proj_df)
    top_df["Div_Anual_Estimado"] = top_df["Preco"] * top_df["DY"] / 100

    # Formatar tabela de exibição
    display_df = top_df[[
        "Segmento", "Preco", "P/VP", "DY", "Vacancia", "Liquidez",
        "Div_Anual_Estimado", "Adequação", "Preco_Projetado",
        "Retorno_Esperado", "Prob_Queda"
    ]].copy()
    display_df["Liquidez"] = display_df["Liquidez"].apply(lambda x: f"R$ {x:.2f} mi")
    display_df["DY"] = display_df["DY"].apply(lambda x: f"{x:.1f}%")
    display_df["Vacancia"] = display_df["Vacancia"].apply(lambda x: f"{x:.1f}%")
    display_df["Div_Anual_Estimado"] = display_df["Div_Anual_Estimado"].apply(lambda x: f"R$ {x:.2f}")
    display_df["Adequação"] = display_df["Adequação"].apply(lambda x: f"{x:.2%}")
    display_df["Retorno_Esperado"] = display_df["Retorno_Esperado"].apply(lambda x: f"{x:+.2%}")
    display_df["Prob_Queda"] = display_df["Prob_Queda"].apply(lambda x: f"{x:.2%}")
    display_df["Preco_Projetado"] = display_df["Preco_Projetado"].apply(lambda x: f"R$ {x:.2f}")

    # Exibição principal
    if run_button:
        st.success(f"Análise concluída para **{profile}**, **{objective}**, horizonte **{horizon}**.")
        st.dataframe(display_df, use_container_width=True)

        # Salvar interação (cada linha do ranking é uma recomendação)
        now = pd.Timestamp.now()
        target_date = now + pd.DateOffset(months=horizon_months)
        for t in top_df.index:
            row = {
                "Timestamp": now,
                "Ticker": t,
                "Capital": capital,
                "Profile": profile,
                "Objective": objective,
                "Horizon_Months": horizon_months,
                "Price_At_Rec": top_df.loc[t, "Preco"],
                "Date_At_Rec": now,
                "Target_Date": target_date,
                "Predicted_Prob": top_df.loc[t, "Base_Prob"],
                "Recommended": True,
                "Actual_Return": np.nan,
                "Risk_Free_Return": np.nan,
                "Outcome": np.nan,
                "Evaluated": False,
            }
            interactions = pd.concat([interactions, pd.DataFrame([row])], ignore_index=True)
        save_interactions(interactions)
        st.info("Suas recomendações foram armazenadas e serão avaliadas após o prazo do horizonte.")

    # Tabs para detalhes e simulação
    tab1, tab2, tab3 = st.tabs(["📊 Ranking", "🔍 Detalhes", "📈 Simulação"])

    with tab1:
        if run_button:
            st.dataframe(display_df, use_container_width=True)
        else:
            st.info("Clique em **Executar Análise** para gerar o ranking.")

    with tab2:
        fii = st.selectbox("Selecione um FII para ver detalhes", current_df.index.tolist())
        if fii:
            r = current_df.loc[fii]
            st.markdown(f"### Detalhes do FII {fii}")
            st.write(f"**Segmento:** {r['Segmento']}")
            st.write(f"**Preço atual:** R$ {r['Preco']:.2f}")
            st.write(f"**P/VP:** {r['P/VP']:.2f}")
            st.write(f"**Dividend Yield:** {r['DY']:.1f}%")
            st.write(f"**Dividendo Anual Estimado:** R$ {r['Preco'] * r['DY'] / 100:.2f}")
            st.write(f"**Vacância:** {r['Vacancia']:.1f}%")
            st.write(f"**Liquidez diária:** R$ {r['Liquidez']:.2f} milhões")
            st.write(f"**Volatilidade anualizada:** {r['Volatilidade']:.2%}")
            st.write(f"**Momentum 3m:** {r['Momentum']:.2%}")
            st.write(f"**Probabilidade base (BN+fundamentalista):** {r['Base_Prob']:.2%}")

    with tab3:
        fii_sim = st.selectbox("Selecione um FII para simular", current_df.index.tolist(), key="sim_select")
        if fii_sim:
            r = current_df.loc[fii_sim]
            price = r["Preco"]
            vol = r["Volatilidade"]
            bp = r["Base_Prob"]
            months = horizon_months
            exp_price, ret_esp, prob_neg = project_price_monte_carlo(price, bp, vol, months=months)
            sigma_m = vol / np.sqrt(12)
            drift_annual = (bp - 0.5) * 0.30
            drift_m = drift_annual / 12
            scen = {"prob": 0.35, "drift": 0.010}
            rng = np.random.default_rng(42)
            z = rng.normal(size=(1000, months))
            rets = drift_m + scen["drift"] + sigma_m * z
            final_prices = price * np.exp(np.cumsum(rets, axis=1))[:, -1]

            fig, ax = plt.subplots(figsize=(8, 4))
            ax.hist(final_prices, bins=50, alpha=0.7, color="#4da6ff")
            ax.axvline(price, color="red", linestyle="--", label="Preço atual")
            ax.axvline(exp_price, color="green", linestyle="--", label=f"Projeção média ({horizon})")
            ax.set_title(f"Distribuição de preços em {months} meses - {fii_sim} (cenário bull)")
            ax.set_xlabel("Preço projetado (R$)")
            ax.set_ylabel("Frequência")
            ax.legend()
            st.pyplot(fig)

In [ ]:
# @title Bloco 9: Execução do aplicativo

if __name__ == "__main__":
    main()